# 08 · Repairing the sensitivity fault (Section 8)

**Question (RQ3).** Can training remove F2?

**What we did.** Small low-rank adapters (LoRA) were trained on COMET-22's encoder with each segment shown under many cuts, all with its one human rating. Five folds: one Indic language held out each time.
* **A1 (plain):** fits the human rating only.
* **A2 (anchored):** also stays close to the frozen original metric.

The adapter update is scaled by a weight lambda (0 = original metric, 1 = full adapter), chosen on the four training languages only. *Repair* is the gain in agreement at the most fragmented bucket; *cost* is the loss on ordinary text.

Training ran on GPUs (`experiments/repair/`); its evaluations are in `data/derived/repair/`. This notebook rebuilds the tables of Section 8 (about a minute).

In [ ]:
import os
import sys
import warnings
from pathlib import Path

# quieten library notices that do not affect any result
os.environ["TRANSFORMERS_VERBOSITY"] = "error"      # "no PyTorch found": no model is needed here
warnings.filterwarnings("ignore", module="tqdm")
warnings.filterwarnings("ignore", module="transformers")

import pandas as pd

ROOT = Path.cwd().resolve().parent          # the repository root
sys.path.insert(0, str(ROOT / "src"))
TABLES = ROOT / "results" / "tables"
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
pd.set_option("display.precision", 3)

def table(name):
    """A table written by the analysis, from results/tables/."""
    return pd.read_csv(TABLES / f"{name}.csv")

In [ ]:
from tokinv import repair
repair.run()

## A1 on held-out languages: agreement per bucket, before and after

In [ ]:
table("lora_reseg_dose_response_lolo")

## A2: lambda chosen on the training languages, then measured on the held-out one

In [ ]:
table("nested_lambda_selection")[["held_out", "lambda_star", "heldout_gain", "heldout_cost", "heldout_retention_pct"]]

## Does the result depend on the settings? (App. Hyperparameter Frontier)

Mean repair for every LoRA rank and loss weight at lambda = 0.5, and the selection of the loss weights by the same rule as lambda.

In [ ]:
table("frontier_grid_lambda0.5")

In [ ]:
table("mu_nested_selection")[["outer", "mu_sd", "mu_an", "lambda", "heldout_gain", "heldout_cost", "fixed_heldout_gain", "fixed_heldout_cost"]]

## Transfer to scripts the adapter never saw

A2, trained without Gujarati, applied unchanged to six WMT settings. Repair is positive in five of six; Arabic moves from -0.408 to -0.069.

In [ ]:
table("transfer_selected_lambda")